# MATRIX Latency Benchmark — Analysis

Neutral analysis of **one** latency benchmark CSV produced by `latency.py`.

This notebook **reports the latency as measured**, from several angles. It draws
no thresholds, no "acceptable" lines, and states no verdict. The plots show the
data straight — whether the middleware overhead is small or large — and you draw
the conclusions.

**To use:** set `CSV_SUFFIX` below to the filename printed by `latency.py`
(without the directory), then *Run All*.


In [3]:
# --- Configuration -------------------------------------------------------
# Paste the CSV filename printed by latency.py (WITHOUT the directory path).
CSV_SUFFIX = "latency_gateway_overhead_v1_20261031_130504.csv"

import os
# The benchmark always writes into this folder, with filenames of the form
#   latency_<suffix>          e.g.  latency_gateway_overhead_v1_20261103_142310.csv
# You provide only the <suffix> above; we prepend "latency_" and the folder.
# (If you paste the full "latency_...csv" name, that's handled too.)
RESULTS_DIR = os.path.join("..","results", "latency-benchmark")
_fname = CSV_SUFFIX if CSV_SUFFIX.startswith("latency_") else f"latency_{CSV_SUFFIX}"
CSV_PATH = os.path.join(RESULTS_DIR, _fname)
FIG_DIR     = os.path.join(RESULTS_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# --- Publication styling (IEEE two-column friendly) ----------------------
mpl.rcParams.update({
    "figure.dpi": 120,      # on-screen preview
    "savefig.dpi": 300,     # saved files (print quality)
    "font.family": "serif", # matches IEEE body text
    "font.size": 9,
    "axes.titlesize": 10, "axes.labelsize": 9, "legend.fontsize": 8,
    "xtick.labelsize": 8, "ytick.labelsize": 8,
    "axes.grid": True, "grid.alpha": 0.3, "savefig.bbox": "tight",
})

COL_W, FULL_W = 3.5, 7.16   # single- and double-column widths (inches)

# Symmetric, colour-blind-safe styling — neither path visually privileged.
PATH_STYLE = {
    "direct":   {"color": "#0072B2", "label": "direct (SCADA\u2192PLC)"},
    "indirect": {"color": "#D55E00", "label": "indirect (SCADA\u2192Pi\u2192PLC)"},
}
PLC_ORDER = ["siemens", "ab", "tm221"]   # fixed order across every figure

def savefig(fig, name):
    # Save each figure as vector PDF (for the paper) and PNG (quick view).
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(FIG_DIR, f"{name}.{ext}"))
    print(f"saved: {name}.pdf / .png")

print("CSV_PATH =", CSV_PATH)

CSV_PATH = ..\results\latency-benchmark\latency_gateway_overhead_v1_20261031_130504.csv


In [4]:
# --- Load and validate ---------------------------------------------------
if not os.path.exists(CSV_PATH):
    # Help find the right file rather than failing blindly.
    avail = sorted(f for f in os.listdir(RESULTS_DIR) if f.endswith(".csv")) \
            if os.path.isdir(RESULTS_DIR) else []
    raise FileNotFoundError(
        f"{CSV_PATH} not found.\nAvailable CSVs in {RESULTS_DIR}:\n  "
        + ("\n  ".join(avail) if avail else "(none)"))

df = pd.read_csv(CSV_PATH)

# Fail loudly if the schema differs from what latency.py writes.
EXPECTED = ["timestamp_utc","run_label","path","protocol","plc","operation",
            "phase","sample_index","rtt_ms","success","error","counter",
            "inter_sample_delay_ms"]
missing = [c for c in EXPECTED if c not in df.columns]
assert not missing, f"CSV missing columns: {missing}"

# rtt_ms is blank on failed samples -> NaN; success 0/1 -> bool.
df["rtt_ms"]  = pd.to_numeric(df["rtt_ms"], errors="coerce")
df["success"] = df["success"].astype(int).astype(bool)

print(f"loaded {len(df):,} rows from {os.path.basename(CSV_PATH)}")
print("paths :", sorted(df["path"].unique()))
print("plcs  :", sorted(df["plc"].unique()))
print("phases:", sorted(df["phase"].unique()))
df.head()

loaded 5,100 rows from latency_gateway_overhead_v1_20261031_130504.csv
paths : ['direct', 'indirect']
plcs  : ['ab', 'siemens', 'tm221']
phases: ['measure', 'warmup']


,timestamp_utc,run_label,path,protocol,plc,operation,phase,sample_index,rtt_ms,success,error,counter,inter_sample_delay_ms
0,2026-10-31T11:05:05.011170+00:00,gateway_overhead_v1,direct,opcua,siemens,read,warmup,0,91.0435,True,NaN,NaN,50
1,2026-10-31T11:05:05.156594+00:00,gateway_overhead_v1,direct,opcua,siemens,read,warmup,1,NaN,False,ConnectionError('Connection is closed'),NaN,50
2,2026-10-31T11:05:05.240138+00:00,gateway_overhead_v1,direct,opcua,siemens,read,warmup,2,NaN,False,ConnectionError('client is disconnected'),NaN,50
3,2026-10-31T11:05:05.291830+00:00,gateway_overhead_v1,direct,opcua,siemens,read,warmup,3,NaN,False,ConnectionError('client is disconnected'),NaN,50
4,2026-10-31T11:05:05.356532+00:00,gateway_overhead_v1,direct,opcua,siemens,read,warmup,4,NaN,False,ConnectionError('client is disconnected'),NaN,50


## Sample inventory & failures

Reported explicitly so nothing is hidden. Failed samples (timeouts/errors) are
counted here and then excluded from the statistics below — but you can see the
failure rate, which matters: percentiles computed over successes alone would be
misleading if the failure rate were high.

In [5]:
# --- Counts per group/phase (confirms the run was complete) --------------
inv = (df.groupby(["path","plc","phase"]).size().rename("n").reset_index()
         .pivot_table(index=["path","plc"], columns="phase", values="n", fill_value=0))
print("Sample counts:")
display(inv)

# --- Measure-phase failure rate (warmup is excluded from stats everywhere) -
meas = df[df["phase"] == "measure"]
fails = (meas.assign(fail=~meas["success"]).groupby(["path","plc"])
             .agg(n=("success","size"), failures=("fail","sum")))
fails["failure_rate_%"] = (100 * fails["failures"] / fails["n"]).round(2)
print("\nMeasure-phase failures:")
display(fails)

Sample counts:


phase             measure  warmup
path     plc                     
direct   ab        1000.0    20.0
         siemens   1000.0    20.0
         tm221     1000.0    20.0
indirect ab        1000.0    20.0
         siemens   1000.0    20.0


Measure-phase failures:


n  failures  failure_rate_%
path     plc                                    
direct   ab       1000         0             0.0
         siemens  1000      1000           100.0
         tm221    1000         0             0.0
indirect ab       1000       976            97.6
         siemens  1000         0             0.0

In [6]:
# --- Working set: measure-phase, successful, finite RTTs ------------------
# Every statistic/plot below uses this. Warmup and failures are excluded from
# stats (failures were quantified above, so nothing is silently discarded).
ok = df[(df["phase"] == "measure") & df["success"] & df["rtt_ms"].notna()].copy()
present_paths = [p for p in ["direct","indirect"] if p in ok["path"].unique()]
plcs = [p for p in PLC_ORDER if p in ok["plc"].unique()]
print(f"{len(ok):,} measure-phase successful samples used for statistics")
print("paths present:", present_paths, "| plcs present:", plcs)

3,024 measure-phase successful samples used for statistics
paths present: ['direct', 'indirect'] | plcs present: ['siemens', 'ab', 'tm221']


## Summary statistics

The numeric ground truth (all in ms). Percentiles use linear interpolation
(numpy default), matching `latency.py`'s own summary. Exported to
`summary_stats.csv` / `.tex` for direct use in the paper.

In [7]:
# --- Per (path, plc) summary --------------------------------------------
def describe(g):
    r = g["rtt_ms"].values
    return pd.Series({
        "n": len(r), "min": np.min(r), "median": np.median(r), "mean": np.mean(r),
        "p95": np.percentile(r, 95), "p99": np.percentile(r, 99), "max": np.max(r),
        "std": np.std(r, ddof=1) if len(r) > 1 else 0.0,
        "IQR": np.percentile(r, 75) - np.percentile(r, 25),
    })

summary = ok.groupby(["path","plc"]).apply(describe).round(3).reset_index()
# Stable ordering: by PLC then path.
summary["plc"] = pd.Categorical(summary["plc"], PLC_ORDER, ordered=True)
summary = summary.sort_values(["plc","path"]).reset_index(drop=True)
print("All values in ms.")
display(summary)

summary.to_csv(os.path.join(FIG_DIR, "summary_stats.csv"), index=False)
try:
    with open(os.path.join(FIG_DIR, "summary_stats.tex"), "w") as f:
        f.write(summary.to_latex(index=False, float_format="%.3f"))
    print("saved: summary_stats.csv / .tex")
except Exception as e:
    print("summary_stats.csv saved; .tex skipped:", e)

All values in ms.


,path,plc,n,min,median,mean,p95,p99,max,std,IQR
0,indirect,siemens,1000.0,47.374,64.751,66.649,96.802,112.626,142.014,14.593,2.028
1,direct,ab,1000.0,2.341,6.182,6.110,8.546,10.196,11.253,1.584,1.911
2,indirect,ab,24.0,3.848,17.546,19.056,30.779,44.726,48.538,7.660,0.752
3,direct,tm221,1000.0,1.382,2.548,2.554,3.120,3.429,4.547,0.376,0.530


saved: summary_stats.csv / .tex


## Distribution shape — ECDF

Empirical cumulative distribution of RTT, direct vs indirect, one panel per PLC.
Read any percentile off the y-axis: pick a y value (e.g. 0.95), read across to a
curve, and the x where it crosses is that path's p95. The horizontal gap between
the two curves at a given y is the middleware overhead **at that percentile**.

Shown on a **linear** x-axis (absolute magnitude — a large overhead looks large)
and a **log** x-axis (shape across scales). Dotted guides mark the 50/95/99th
percentiles — statistical reading aids, not thresholds.

In [ ]:
def ecdf(x):
    xs = np.sort(x)
    ys = np.arange(1, len(xs) + 1) / len(xs)   # cumulative proportion
    return xs, ys

def plot_ecdf(xscale):
    fig, axes = plt.subplots(1, len(plcs), figsize=(FULL_W, 2.4), sharey=True)
    if len(plcs) == 1: axes = [axes]
    for ax, plc in zip(axes, plcs):
        for path in present_paths:
            sub = ok[(ok["plc"] == plc) & (ok["path"] == path)]
            if sub.empty: continue
            xs, ys = ecdf(sub["rtt_ms"].values)
            ax.step(xs, ys, where="post", lw=1.3,
                    color=PATH_STYLE[path]["color"], label=PATH_STYLE[path]["label"])
        for q in (0.5, 0.95, 0.99):           # percentile reading guides (neutral)
            ax.axhline(q, ls=":", lw=0.6, color="grey", alpha=0.6)
        ax.set_title(plc); ax.set_xlabel("RTT (ms)"); ax.set_xscale(xscale)
    axes[0].set_ylabel("cumulative proportion")
    axes[0].legend(loc="lower right")
    fig.suptitle(f"Latency ECDF by PLC ({xscale} x-axis)", y=1.03)
    savefig(fig, f"ecdf_{xscale}")
    plt.show()

plot_ecdf("linear")
plot_ecdf("log")

## Spread and outliers — box plots

Box = median and interquartile range; whiskers = 1.5\u00d7IQR; points beyond are
outliers, shown individually so the tail is not hidden. Linear y shows absolute
magnitude; log y shows spread for right-skewed distributions.

In [ ]:
from matplotlib.patches import Patch

def plot_box(yscale):
    fig, ax = plt.subplots(figsize=(FULL_W, 3.0))
    positions, data, colors, tick_pos, ticks = [], [], [], [], []
    pos = 0
    for plc in plcs:
        centre = []
        for path in present_paths:
            vals = ok[(ok["plc"] == plc) & (ok["path"] == path)]["rtt_ms"].values
            pos += 1
            positions.append(pos); data.append(vals)
            colors.append(PATH_STYLE[path]["color"]); centre.append(pos)
        tick_pos.append(np.mean(centre)); ticks.append(plc)
        pos += 1   # blank slot between PLC groups
    bp = ax.boxplot(data, positions=positions, widths=0.7, patch_artist=True,
                    flierprops=dict(marker=".", markersize=3, alpha=0.4),
                    medianprops=dict(color="black"))
    for patch, c in zip(bp["boxes"], colors):
        patch.set_facecolor(c); patch.set_alpha(0.5)
    ax.set_xticks(tick_pos); ax.set_xticklabels(ticks)
    ax.set_ylabel("RTT (ms)"); ax.set_yscale(yscale)
    ax.legend(handles=[Patch(facecolor=PATH_STYLE[p]["color"], alpha=0.5,
                             label=PATH_STYLE[p]["label"]) for p in present_paths],
              loc="upper left")
    ax.set_title(f"RTT distribution by PLC and path ({yscale} y-axis)")
    savefig(fig, f"box_{yscale}")
    plt.show()

plot_box("linear")
plot_box("log")

## Measured overhead — indirect minus direct

The middleware's added latency, stated plainly as the difference in median and
p95 between the two paths, per PLC. These are measured quantities (\u0394 =
indirect \u2212 direct), reported without judgement. Only shown if both paths are
present in the data.

In [ ]:
if set(["direct","indirect"]).issubset(present_paths):
    piv = summary.pivot(index="plc", columns="path", values=["median","p95"])
    rows = []
    for plc in plcs:
        rows.append({
            "plc": plc,
            "direct_median":   piv.loc[plc, ("median","direct")],
            "indirect_median": piv.loc[plc, ("median","indirect")],
            "delta_median":    piv.loc[plc, ("median","indirect")] - piv.loc[plc, ("median","direct")],
            "direct_p95":      piv.loc[plc, ("p95","direct")],
            "indirect_p95":    piv.loc[plc, ("p95","indirect")],
            "delta_p95":       piv.loc[plc, ("p95","indirect")] - piv.loc[plc, ("p95","direct")],
        })
    overhead = pd.DataFrame(rows).round(3)
    display(overhead)
    overhead.to_csv(os.path.join(FIG_DIR, "overhead.csv"), index=False)

    fig, axes = plt.subplots(1, 2, figsize=(FULL_W, 2.8))
    for ax, stat in zip(axes, ["median","p95"]):
        x = np.arange(len(plcs)); w = 0.38
        dvals = [piv.loc[p, (stat,"direct")]   for p in plcs]
        ivals = [piv.loc[p, (stat,"indirect")] for p in plcs]
        ax.bar(x - w/2, dvals, w, color=PATH_STYLE["direct"]["color"],   alpha=0.85, label="direct")
        ax.bar(x + w/2, ivals, w, color=PATH_STYLE["indirect"]["color"], alpha=0.85, label="indirect")
        for xi, p in zip(x, plcs):     # neutral numeric delta above each group
            d = piv.loc[p, (stat,"indirect")] - piv.loc[p, (stat,"direct")]
            ax.text(xi, max(piv.loc[p,(stat,"direct")], piv.loc[p,(stat,"indirect")]),
                    f"\u0394={d:+.2f}", ha="center", va="bottom", fontsize=7)
        ax.set_xticks(x); ax.set_xticklabels(plcs)
        ax.set_title(f"{stat} RTT"); ax.set_ylabel("ms")
    axes[0].legend(loc="upper left")
    fig.suptitle("Direct vs indirect per PLC (\u0394 = indirect \u2212 direct)", y=1.03)
    savefig(fig, "overhead_median_p95")
    plt.show()
else:
    print("Both paths not present in this CSV \u2014 overhead comparison skipped.")

## Stability over the run — RTT vs sample index

Each sample in order, to reveal drift, periodic spikes, or warm-up effects.
Warm-up samples (excluded from statistics) are drawn faintly, at negative
indices, before the x=0 boundary, so you can see *why* they are excluded. A flat
cloud means a stationary distribution; a trend or periodic spikes would show
here.

In [ ]:
fig, axes = plt.subplots(len(plcs), 1, figsize=(FULL_W, 2.0 * len(plcs)))
if len(plcs) == 1: axes = [axes]
for ax, plc in zip(axes, plcs):
    for path in present_paths:
        col = PATH_STYLE[path]["color"]
        # measured samples (bold), at their own index 0..N-1
        m = df[(df["plc"]==plc) & (df["path"]==path) & (df["phase"]=="measure") & df["success"]]
        ax.scatter(m["sample_index"], m["rtt_ms"], s=4, alpha=0.4, color=col,
                   label=f"{path} (measure)")
        # warmup samples (faint), shifted to negative indices so they sit before x=0
        w = df[(df["plc"]==plc) & (df["path"]==path) & (df["phase"]=="warmup") & df["success"]]
        if not w.empty:
            W = int(w["sample_index"].max()) + 1
            ax.scatter(w["sample_index"] - W, w["rtt_ms"], s=10, alpha=0.5,
                       marker="x", color=col)
    ax.axvline(0, ls="--", lw=0.6, color="grey", alpha=0.7)  # warmup | measure boundary
    ax.set_title(f"{plc} \u2014 RTT vs sample index")
    ax.set_xlabel("sample index  (negative = warm-up)"); ax.set_ylabel("RTT (ms)")
axes[0].legend(loc="upper right", markerscale=2)
fig.tight_layout()
savefig(fig, "sequence_rtt")
plt.show()

## Outputs

Figures saved to `results/latency-benchmark/figures/` as **PDF** (vector, for the
paper) and **PNG**. Summary statistics in `summary_stats.csv` / `.tex`; the
overhead table in `overhead.csv`.

All plots report the measured latency without thresholds or verdicts \u2014 the
interpretation is yours.